# 11주차 ① 셀프 어텐션 직접 구현 — 실습 1~2  〔빈칸본〕

> **빈칸이 2곳입니다.** 셀 5 의 `scores`·`alpha` 두 줄이고,
> **이 두 줄이 트랜스포머의 심장**입니다. 전체를 타이핑하지 말고 **여기에만 집중**하세요.
> 다 채운 노트북은 `20_self_attention.ipynb` 로 저장합니다.

**목표**: 순환을 없앴을 때 **얻는 것과 잃는 것**을 이해하고,
`nn.MultiheadAttention` 을 **부르지 않고** 스케일드 닷-프로덕트 어텐션을 행렬곱으로 직접 짜며,
`√d_k` 로 나누는 이유를 **수치로** 확인한다.

> **GPU 불필요** — 오늘은 학습을 하지 않습니다. 노트북 PC로도 전부 됩니다.
> **작은 숫자(B=1, T=4, C=8)로 시작합니다.** 큰 텐서는 shape 만 보이고 값이 안 보여
> 이해에 도움이 되지 않습니다.

> **오늘 완전히 새로운 것은 Q·K·V 발상 하나뿐입니다.**
> softmax 는 5주차, 행렬곱과 `transpose` 는 3주차, 잔차 연결은 7주차에 이미 했습니다.
> **여러분은 트랜스포머 부품을 이미 다 갖고 있습니다.**

### 순환을 없앤다는 발상 ★

```
   LSTM :  x₁ → x₂ → x₃ → ... → x₄₀
           └───순서대로. 앞이 끝나야 뒤를 계산───┘
     ① 병렬화가 안 된다     40 토큰이면 40번을 기다려야 한다  ← GPU 가 노는 시간
     ② 멀수록 희미하다      1번 토큰의 정보가 40번까지 가려면 40번을 거친다

   셀프 어텐션 :   모든 위치가  모든 위치를  직접 본다
        x₁ ←──┬──→ x₂
         ↕    │     ↕            거리 1 이든 39 든 "한 번"에 닿는다
        x₃ ←──┴──→ x₄            그리고 전부 동시에 계산된다 (행렬곱 한 번)
```

| | 얻은 것 | 잃은 것 |
|---|---|---|
| **병렬화** | 전 토큰을 행렬곱 한 번으로 | — |
| **거리** | 멀어도 한 번에 닿는다 | — |
| **순서** | — | **순서 정보가 사라진다** ★ |
| **계산량** | — | 토큰 수의 **제곱**(T×T 행렬) |

> **핵심 메시지 ★★ (기말 출제 1순위)**: 순환을 버려서 **병렬화와 긴 의존성**을 얻고,
> 대신 **순서 정보**를 잃습니다. 잃은 것은 **2교시에 위치 인코딩으로 되돌려 줍니다.**

### Q·K·V — 세 가지 역할 ★★

```
   유튜브에서 영상을 찾는다
     Query  (질의)   "내가 지금 찾는 것"          ← 검색창에 친 말
     Key    (열쇠)   "각 영상이 내건 간판"        ← 제목·태그
     Value  (값)     "실제로 가져올 내용"         ← 영상 본문

   입력 x  (B, T, C)
     ├── W_q ─→  Q     "나는 무엇을 찾는가"
     ├── W_k ─→  K     "나는 무엇으로 매칭되는가"
     └── W_v ─→  V     "나는 무엇을 내어 줄 것인가"

   ※ 같은 x 에서 셋이 나온다  →  그래서 "셀프(self)" 어텐션 ★
```

```
                       Q Kᵀ
   Attention(Q,K,V) = softmax( ──── ) V
                        √d_k
```

> **핵심 메시지 ★★ (기말 출제 1순위)**: *"Q·K·V가 각각 무엇인가"* 를
> **무엇을 찾는가 / 무엇으로 매칭되는가 / 무엇을 가져오는가** 로 답할 수 있어야 합니다.
> 수식은 못 외워도 이 세 문장은 외우세요.

## 실습 1 — Q·K·V 선형 투영 만들기

In [ ]:
# 셀 1 — 작은 텐서로 시작한다
import torch, torch.nn as nn, torch.nn.functional as F
import math
torch.manual_seed(42)                    # 6주차 재현성 ★

B, T, C = 1, 4, 8                        # 배치 1, 토큰 4개, 채널 8
x = torch.randn(B, T, C)                 # (B, T, C) = (1, 4, 8)
print("입력 x :", x.shape)

# 토큰 4개에 이름을 붙여 두면 나중에 어텐션 맵이 읽힌다
tokens = ["나는", "어제", "영화를", "봤다"]
print("토큰 :", tokens)

In [ ]:
# 셀 2 — 세 개의 선형층
d_k = 8                                  # Q·K 의 차원 (여기서는 C 와 같게)

W_q = nn.Linear(C, d_k, bias=False)      # (B,T,C) → (B,T,d_k)
W_k = nn.Linear(C, d_k, bias=False)
W_v = nn.Linear(C, d_k, bias=False)

Q = W_q(x)                               # (1, 4, 8)
K = W_k(x)                               # (1, 4, 8)
V = W_v(x)                               # (1, 4, 8)

print("Q :", Q.shape, "| K :", K.shape, "| V :", V.shape)
print("\nW_q 의 가중치 shape :", W_q.weight.shape, " ← 학습 대상 ★")

> **관찰 포인트 ★**: Q·K·V 를 만드는 것은 **그냥 선형층 3개**입니다.
> 대단한 게 아닙니다. **학습되는 것은 이 `W_q`, `W_k`, `W_v` 세 행렬**이고,
> *"무엇을 찾고 무엇으로 매칭될지"* 를 모델이 스스로 정하게 하는 장치입니다.

In [ ]:
# 셀 3 — bias=False 인 이유 · 파라미터 수
total = sum(p.numel() for p in [W_q.weight, W_k.weight, W_v.weight])
print(f"Q·K·V 투영 파라미터 : {total:,} 개  (= 3 × {C} × {d_k})")
print("bias 는 관례적으로 생략한다 (있어도 무방, 성능 차이 미미)")

# 세 개가 정말 다른 값인지 확인
print("\nQ[0,0] :", Q[0, 0].detach().numpy().round(3))
print("K[0,0] :", K[0, 0].detach().numpy().round(3), " ← 같은 토큰인데 다른 벡터 ★")

> **핵심 메시지**: **같은 토큰이 Q 로서와 K 로서 다른 벡터**가 됩니다.
> *"내가 찾는 것"* 과 *"내가 내거는 간판"* 은 다른 것이니까요.
> 여기서 `W_q = W_k` 로 두면 어텐션이 **자기 자신에만 쏠립니다.**

## 실습 2 — 스케일드 닷-프로덕트 직접 구현 ★★

In [ ]:
# 셀 4 — 1단계 : 모든 토큰 쌍의 유사도 QKᵀ
# K 를 전치한다: (B, T, d_k) → (B, d_k, T)
scores = Q @ K.transpose(-2, -1)         # (B,T,d_k) @ (B,d_k,T) → (B, T, T) ★
print("scores :", scores.shape, " ← (B, T, T) : 토큰 4개 × 토큰 4개")
print(scores[0].detach().numpy().round(2))

```
   scores[0] 을 읽는 법  ★

              나는   어제  영화를  봤다      ← Key   (누구를 보는가)
      나는  [  2.1   -0.3   0.8   1.2 ]
      어제  [ -0.3    1.9   0.1  -0.5 ]
    영화를  [  0.8    0.1   2.4   0.9 ]
      봤다  [  1.2   -0.5   0.9   1.7 ]
        ↑
      Query (누가 보는가)

   i 행 j 열 = "i 번 토큰이 j 번 토큰을 얼마나 볼 것인가"의 원점수
```

> **핵심 메시지 ★ (출제 지점)**: 어텐션 행렬의 shape 은 **`(T, T)`** 입니다.
> 토큰 수의 **제곱**이고, 그래서 셀프 어텐션의 계산 복잡도가 **시퀀스 길이의 제곱**입니다.
> 문장이 2배 길어지면 계산은 **4배**가 됩니다. (긴 문서 처리가 어려운 이유)

In [ ]:
# 셀 5 — 2단계 : √d_k 로 나누고 확률로 만든다
# ───── 빈칸 ① : QKᵀ 를 √d_k 로 나눈다 (한 줄) ─────
# 힌트:  scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)


# ───── 빈칸 ② : softmax 로 확률을 만든다 (한 줄) ─────
# 힌트:  alpha = torch.softmax(scores, dim=__)      ← 어느 축이어야 하나?


print("alpha :", alpha.shape)
print(alpha[0].detach().numpy().round(3))
print("\n각 행의 합 :", alpha[0].sum(dim=-1).detach().numpy().round(4), " ← 전부 1.0 ★")
print("각 열의 합 :", alpha[0].sum(dim=-2).detach().numpy().round(4), " ← 1 이 아니다")

> **관찰 포인트 ★★ (출제 지점)**: `dim=-1` 로 softmax 하는 이유는
> **"한 토큰이 다른 토큰들에게 나눠 주는 비중"** 이 합쳐서 1이어야 하기 때문입니다.
> **각 행의 합이 1** 입니다. **열의 합은 1이 아닙니다.**

> ⚠️ **자주 나오는 실수**: `dim=0` 이나 `dim=1` 로 softmax 하는 것.
> 값은 나오지만 **의미가 완전히 다릅니다.** `dim=-1` 을 습관화하세요.

In [ ]:
# 셀 6 — 3단계 : V 를 비중대로 섞는다
out = alpha @ V                          # (B,T,T) @ (B,T,d_k) → (B, T, d_k) ★
print("출력 :", out.shape, " ← 입력과 같은 (B, T, ·) 로 돌아왔다")

# 첫 토큰의 출력은 정말 V 들의 가중합인가? 손으로 검증한다
manual = sum(alpha[0, 0, j] * V[0, j] for j in range(T))
print("\n검증 :", torch.allclose(out[0, 0], manual, atol=1e-5), " ← True 여야 한다 ★")

In [ ]:
# 셀 7 — 재사용 가능한 함수로 묶는다 ★
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Q,K,V: (..., T, d_k)  →  out: (..., T, d_k), alpha: (..., T, T)"""
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)     # (..., T, T)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))   # 3교시에서 쓴다
    alpha = torch.softmax(scores, dim=-1)                 # (..., T, T)
    return alpha @ V, alpha                               # (..., T, d_k), (..., T, T)

out, alpha = scaled_dot_product_attention(Q, K, V)
print(out.shape, alpha.shape)

> **핵심 메시지 ★★**: **방금 여러분이 트랜스포머의 심장을 짰습니다.**
> GPT 도 BERT 도 이 함수를 부릅니다. **본질은 세 줄**입니다 —
> 곱하고(`Q @ K.T`), 나누고(`/√d_k`), 확률로 만들어(`softmax`) 섞는다(`@ V`).

> ⚠️ **`mask == 0` 자리에 `-inf` 를 넣는 이유**: softmax 를 통과하면 `e^(-inf) = 0` 이 되어
> **그 위치의 비중이 정확히 0** 이 됩니다. 10주차 패딩 마스킹과 **같은 기법**입니다.

## `√d_k` 로 나누는 이유 ★

In [ ]:
# 셀 8 — 차원이 커지면 내적이 커진다 (30초 시연)
for dim in [8, 64, 512]:
    q = torch.randn(1000, dim)
    k = torch.randn(1000, dim)
    dot = (q * k).sum(-1)                       # 내적 1000개
    print(f"d_k={dim:4d} | 내적의 표준편차 {dot.std():6.2f}"
          f"  (√d_k = {math.sqrt(dim):5.2f})")

print("\n→ 내적의 크기는 차원이 커질수록 √d_k 에 비례해 커진다 ★")

In [ ]:
# 셀 9 — 그래서 softmax 가 어떻게 되나
s = torch.tensor([[2.0, 1.0, 0.5, 0.2]])
for scale, name in [(1.0, "d_k=8 수준"), (8.0, "d_k=512 수준(스케일링 없음)")]:
    p = torch.softmax(s * scale, dim=-1)
    print(f"{name:28s} → {p.numpy().round(4)}")

print("\n포화된 softmax 의 기울기도 확인해 보자")
for scale in [1.0, 8.0]:
    z = (s * scale).clone().requires_grad_(True)
    torch.softmax(z, dim=-1)[0, 0].backward()
    print(f"  scale={scale:4.1f} | 기울기 크기 {z.grad.abs().sum().item():.6f}")

```
   d_k=8 수준                   → [0.475 0.175 0.106 0.079]   ← 골고루 본다
   d_k=512 수준(스케일링 없음)  → [0.9997 0.0003 0.0000 0.0000]  ← 한 곳에 쏠린다 ★
```

> **핵심 메시지 ★★ (기말 출제 지점)**:
> ① 차원 `d_k` 가 커지면 내적의 **분산이 커진다**(≈ `d_k` 에 비례).
> ② 큰 값이 softmax 에 들어가면 **한쪽으로 포화**된다(거의 원-핫).
> ③ 포화된 softmax 는 **기울기가 거의 0** 이라 **학습이 멈춘다.**
> → 그래서 `√d_k` 로 나눠 **분산을 1 수준으로 되돌린다.**

| 흔한 오해 | 사실 |
|---|---|
| "값을 작게 만들려고" | 목적은 크기가 아니라 **분산의 정규화** |
| "`d_k` 로 나눠도 되지 않나" | 분산이 `d_k` 에 비례하므로 **표준편차는 `√d_k`**. 그래서 루트 |

---

### 이 노트북 체크리스트

- [ ] 순환을 없애서 **얻는 것 2가지, 잃는 것 1가지**를 말할 수 있다 ★★
- [ ] Q·K·V 를 **자기 말로** 한 문장씩 설명할 수 있다 ★★
- [ ] Q·K·V 가 **같은 x 에서 선형층 3개로** 만들어지는 것을 확인했다
- [ ] 어텐션 행렬이 `(T, T)` 이고 **각 행의 합이 1** 인 것을 확인했다 ★
- [ ] `softmax(dim=-1)` 인 이유를 안다
- [ ] `√d_k` 로 나누는 이유를 **분산 → 포화 → 기울기** 순서로 설명할 수 있다 ★★
- [ ] 셀프 어텐션의 복잡도가 시퀀스 길이의 제곱인 이유를 안다